In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
movie_df = (
    spark.table("databricks_ws.bronze.movies")
    .filter(col("file_date") == file_date)
    )

In [0]:
movies_selected_df = (
    movie_df
    .select(
        col("movieId").cast("int"),
        col("title").cast("string"),
        col("budget").cast("double"),
        col("popularity").cast("double"),
        col("yearReleaseDate").cast("int"),
        to_date(col("releaseDate"), "yyyy-MM-dd").alias("releaseDate"),
        col("revenue").cast("double"),
        col("durationTime").cast("int"),
        col("voteAverage").cast("double"),
        col("voteCount").cast("int"),
        col("file_date")
    )
)

In [0]:
movies_final_df = (
    add_ingestion_date(movies_selected_df)
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("yearReleaseDate", "year_release_date")
    .withColumnRenamed("releaseDate", "release_date")
    .withColumnRenamed("durationTime", "duration_time")
    .withColumnRenamed("voteAverage", "vote_average")
    .withColumnRenamed("voteCount", "vote_count")
    .withColumn("environment", lit("Production"))   
    )

In [0]:
table_name = "databricks_ws.silver.movies"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            movies_final_df.alias("src"),
            "tgt.movie_id = src.movie_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    movies_final_df.write \
        .format("delta") \
        .saveAsTable(table_name)